# 01 - Decision Tree
Validation-based tuning on the shared leakage-aware split. The test partition is evaluated once and saved.

In [ ]:
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, recall_score, ConfusionMatrixDisplay
ROOT=Path.cwd()
if not (ROOT/'eeg_app').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from eeg_app.src.experiments import load_prepared, feature_names, classical_pipeline, save_model_result, experiment_dirs, seed_everything
seed_everything(42)
frame=load_prepared(ROOT); features=feature_names(frame)
train=frame[frame.split.eq('train')]; val=frame[frame.split.eq('validation')]; test=frame[frame.split.eq('test')]
X_train,y_train=train[features],train.label; X_val,y_val=val[features],val.label; X_test,y_test=test[features],test.label
search=ParameterGrid({'model__max_depth':[3,6,10,None],'model__min_samples_leaf':[1,5,20],'model__class_weight':[None,'balanced']})
best=None
for params in search:
    model=classical_pipeline(DecisionTreeClassifier(random_state=42),scale=False).set_params(**params); model.fit(X_train,y_train); pred=model.predict(X_val); key=(f1_score(y_val,pred,zero_division=0),recall_score(y_val,pred,zero_division=0))
    if best is None or key>best[0]: best=(key,params,model)
print('best validation:',best[0],best[1])
model=best[2]; scores=model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else model.predict(X_test)
metrics=save_model_result('decision_tree',model,test,model.predict(X_test),scores,ROOT)
Path(experiment_dirs(ROOT)['results']/'decision_tree_validation.json').write_text(json.dumps({'F1':best[0][0],'Recall':best[0][1],'params':best[1]},indent=2),encoding='utf-8')
print(json.dumps(metrics,indent=2))
ConfusionMatrixDisplay.from_predictions(y_test,model.predict(X_test),display_labels=['normal','seizure']); plt.title('Decision Tree test confusion matrix'); plt.show()